In [ ]:
import os
import json
import glob
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter
from skimage import exposure
from skimage import data, img_as_float
import matplotlib
import numpy as np
matplotlib.rcParams['font.size'] = 8
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import skimage 
from skimage.segmentation import clear_border
from skimage.morphology import white_tophat, disk,remove_small_objects
import matplotlib.patches as mpatches
from skimage.transform import resize
from skimage.filters import threshold_otsu
from skimage.segmentation import active_contour
from skimage.measure import find_contours, label, regionprops
from skimage.draw import polygon


In [ ]:
BASE_DIR = Path(r"c:/Users/ANING/Downloads/54816/54816")

DIR_24   = BASE_DIR / "24_chromosomes_object"
DIR_SNGL = BASE_DIR / "single_chromosomes_object"

ANN_24   = DIR_24 / "annotations"
IMG_24   = DIR_24 / "JEPG"        

ANN_SNGL = DIR_SNGL / "anntations" 
IMG_SNGL = DIR_SNGL / "JEPG"

TRAIN_TXT = BASE_DIR / "train.txt"
TEST_TXT  = BASE_DIR / "test.txt"
DIFF_TXT  = BASE_DIR / "diff_image.txt"

In [ ]:
import random

all_img_24   = sorted(IMG_24.glob("*.jpg"))   + sorted(IMG_24.glob("*.JPG"))
all_img_sngl = sorted(IMG_SNGL.glob("*.jpg")) + sorted(IMG_SNGL.glob("*.JPG"))


SUBSET_SIZE = 100
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
iimg_sngl = random.sample(all_img_sngl, min(SUBSET_SIZE, len(all_img_sngl))) if SUBSET_SIZE else all_img_sngl

print(f"24-chr  subset : {len(img_24)}  / {len(all_img_24)}  total")
print(f"Single  subset : {len(img_sngl)} / {len(all_img_sngl)} total")

In [ ]:
img_raw = skimage.io.imread(img_24[23],as_gray=True)

print("Shape :", img_raw.shape)   
print("dtype :", img_raw.dtype)
print("Range :", img_raw.min().round(3), "–", img_raw.max().round(3))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io

def check_channel_alignment(image_path):

    img = io.imread(image_path)

    if img.ndim != 3 or img.shape[2] != 3:
        print("Image is already single-channel (grayscale).")
        return

    r = img[:,:,0]
    g = img[:,:,1]
    b = img[:,:,2]


    rg_diff = np.mean(np.abs(r - g))
    rb_diff = np.mean(np.abs(r - b))
    gb_diff = np.mean(np.abs(g - b))

    plt.figure(figsize=(8,5))

    plt.hist(r.ravel(), bins=256, alpha=0.5, label="Red")
    # plt.hist(g.ravel(), bins=256, alpha=0.5, label="Green")
    # plt.hist(b.ravel(), bins=256, alpha=0.5, label="Blue")

    plt.xlabel("Pixel Intensity")
    plt.ylabel("Frequency")
    plt.title("RGB Channel Intensity Distributions")
    plt.legend()

    plt.show()


    rg_corr = np.corrcoef(r.flatten(), g.flatten())[0,1]
    rb_corr = np.corrcoef(r.flatten(), b.flatten())[0,1]
    gb_corr = np.corrcoef(g.flatten(), b.flatten())[0,1]


    if rg_corr > 0.99 and rb_corr > 0.99 and gb_corr > 0.99:
        print("\nConclusion: Channels are nearly identical → image is grayscale stored as RGB.")
    else:
        print("\nConclusion: Channels differ → real color or multi-channel image.")

    # visualize channels
    fig, ax = plt.subplots(1,3, figsize=(12,4))

    ax[0].imshow(r, cmap="gray")
    ax[0].set_title("Red channel")

    ax[1].imshow(g, cmap="gray")
    ax[1].set_title("Green channel")

    ax[2].imshow(b, cmap="gray")
    ax[2].set_title("Blue channel")

    for a in ax:
        a.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
img = skimage.filters.gaussian(img_raw, 0.3, preserve_range=True)
img = np.clip(img, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(img.ravel(), bins=256)
axes[0].set_title(f"Grayscale intensity — {img_24[0].stem}")
axes[0].set_xlabel("Pixel intensity (0 = black, 1 = white)")
axes[0].set_ylabel("Frequency")

axes[1].imshow(img, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(img_24[0].stem)
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Contrast stretching
p2, p98 = np.percentile(img, (2, 98))
img_rescale = exposure.rescale_intensity(img, in_range=(p2, p98))

# Equalization
img_eq = exposure.equalize_hist(img)

# Adaptive Equalization
img_adapteq = exposure.equalize_adapthist(img, clip_limit=0.03)

# Display results
fig = plt.figure(figsize=(8, 5))
axes = np.zeros((2, 4), dtype=object)
axes[0, 0] = fig.add_subplot(2, 4, 1)
for i in range(1, 4):
    axes[0, i] = fig.add_subplot(2, 4, 1 + i, sharex=axes[0, 0], sharey=axes[0, 0])
for i in range(0, 4):
    axes[1, i] = fig.add_subplot(2, 4, 5 + i)

ax_img, ax_hist, ax_cdf = plot_img_and_hist(img, axes[:, 0])
ax_img.set_title('Low contrast image')

y_min, y_max = ax_hist.get_ylim()
ax_hist.set_ylabel('Number of pixels')
ax_hist.set_yticks(np.linspace(0, y_max, 5))

ax_img, ax_hist, ax_cdf = plot_img_and_hist(img_rescale, axes[:, 1])
ax_img.set_title('Contrast stretching')

ax_img, ax_hist, ax_cdf = plot_img_and_hist(img_eq, axes[:, 2])
ax_img.set_title('Histogram equalization')

ax_img, ax_hist, ax_cdf = plot_img_and_hist(img_adapteq, axes[:, 3])
ax_img.set_title('Adaptive equalization')

ax_cdf.set_ylabel('Fraction of total intensity')
ax_cdf.set_yticks(np.linspace(0, 1, 5))

# prevent overlap of y-axis labels
fig.tight_layout()
plt.show()

In [ ]:
resized_images = [
    resize(skimage.io.imread(p, as_gray=True), (512,512), 
           preserve_range=True, anti_aliasing=True)
    for p in img_24
]

In [ ]:
mask = chromosome_mask_with_contrast_stretching(img_24[2],gaussian_sigma=0.3,tophat_disk_size=15,block_size=101,opening_disk_size=5,min_object_size=600,)

In [ ]:
mask = chromosome_mask_with_adaptive_histogram(img_24[2],gaussian_sigma=0.3,tophat_disk_size=15,block_size=101,opening_disk_size=5,min_object_size=600,)

In [ ]:
gray_images = [skimage.io.imread(p, as_gray=True) for p in img_24]

In [ ]:
BINS = 256
bin_edges = np.linspace(0, 1, BINS + 1)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2


all_hists = np.stack([
    np.histogram(g.ravel(), bins=bin_edges, density=True)[0]
    for g in gray_images
])

mean_hist = all_hists.mean(axis=0)
std_hist  = all_hists.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(17, 4))

for h in all_hists:
    axes[0].plot(bin_centres, h, color="steelblue", alpha=0.08, linewidth=0.7)
axes[0].plot(bin_centres, mean_hist, color="black", linewidth=1.5, label="mean")
axes[0].set_title("All subset histograms")
axes[0].set_xlabel("intensity")
axes[0].set_ylabel("density")
axes[0].legend()


axes[1].fill_between(bin_centres,
                     np.maximum(mean_hist - std_hist, 0),
                     mean_hist + std_hist,
                     alpha=0.3, color="steelblue", label="±1 std")
axes[1].plot(bin_centres, mean_hist, color="steelblue", linewidth=1.5, label="mean")
axes[1].set_title("Mean histogram ± 1 std")
axes[1].set_xlabel("intensity")
axes[1].legend()



plt.suptitle("Grayscale histogram analysis — 100-image subset", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
plot_segmentation_grid(img_24[:6], chromosome_mask_with_adaptive_histogram)

In [ ]:
plot_segmentation_grid(img_24[:6], chromosome_mask_with_contrast_stretching)

In [ ]:
imgs_with_masks = []
for img_pth in img_24[:3]:
    img = skimage.io.imread(img_pth,as_gray=True)
    mask = chromosome_mask_with_active_contour(img)
    imgs_with_masks.append((img,mask))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

img, mask = imgs_with_masks[0]

color = (0.067, 0.988, 0)  # your RGB color

colored_mask = np.zeros((*mask.shape, 3))
colored_mask[mask > 0] = color

plt.imshow(img, cmap='gray',alpha=1)
plt.imshow(colored_mask, alpha=0.366)

plt.axis('off')
plt.show()

In [ ]:
from skimage.measure import label, regionprops

labels = label(mask)
regions = regionprops(labels)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

img, mask = imgs_with_masks[0]

color = (0.067, 0.988, 0)

# create colored mask
colored_mask = np.zeros((*mask.shape, 3))
colored_mask[mask > 0] = color

fig, ax = plt.subplots()

# show original image
ax.imshow(img, cmap='gray')

# overlay colored mask
ax.imshow(colored_mask, alpha=0.4)

for i, region in enumerate(regions):

    minr, minc, maxr, maxc = region.bbox

    rect = Rectangle(
        (minc, minr),
        maxc - minc,
        maxr - minr,
        fill=False,
        edgecolor=(255/255, 102/255, 0/255),
        linewidth=1
    )

    ax.add_patch(rect)

    # add chromosome label
    ax.text(
        minc,
        minr - 3,
        str(i+1),
        color=(255/255, 255/255, 255/255),
        fontsize=5,
        weight='bold'
    )

ax.axis('off')
plt.show()

In [ ]:
from skimage.measure import label
labels = label(mask)


In [ ]:
img = skimage.io.imread(img_24[2],as_gray=True)

In [ ]:
from skimage.measure import regionprops

regions = regionprops(labels)

chromosomes = []

for region in regions:
    minr, minc, maxr, maxc = region.bbox
    chrom = img[minr:maxr, minc:maxc]
    chromosomes.append((chrom,region))

In [ ]:
chrom,region = chromosomes[1]

In [ ]:
from skimage.transform import rotate
angle   = -region.orientation * 180 / np.pi
rotated = rotate(chrom, angle, resize=True, cval=1, preserve_range=True)
plt.imshow(rotated)
plt.axis("off")


In [ ]:
p2, p98 = np.percentile(rotated, (2, 80))
rotated = exposure.rescale_intensity(rotated, in_range=(p2, p98))
display_image_histogram(rotated)

In [ ]:
from skimage.transform import rotate
import numpy as np

aligned = []

for chrom, region in chromosomes:

    angle   = -region.orientation * 180 / np.pi
    rotated = rotate(chrom, angle, resize=True, cval=1, preserve_range=True)

    aligned.append(rotated)

In [ ]:
from skimage.transform import resize

resized = []

for chrom in aligned:

    r = resize(chrom, (128, 64))

    resized.append(r)

# Band Detection

In [ ]:
n_show = min(6, len(resized))

fig, axes = plt.subplots(n_show, 2, figsize=(10, n_show * 2.5))

for i in range(n_show):
    chrom = resized[i]

    profile              = extract_intensity_profile(chrom)
    smoothed, peaks, props = detect_bands(profile)
    centromere           = find_centromere(chrom)

    # ── left: chromosome image with band and centromere overlays ─────────────
    ax_img = axes[i, 0]
    ax_img.imshow(chrom, cmap='gray')
    for p in peaks:
        ax_img.axhline(p, color='red', linewidth=0.8, alpha=0.7)
    ax_img.axhline(centromere, color='cyan', linewidth=1.2, linestyle='--')
    ax_img.set_title(f"C{i+1} — {len(peaks)} bands detected", fontsize=8)
    ax_img.axis('off')

    # ── right: intensity profile with detected peaks ──────────────────────────
    ax_prof = axes[i, 1]
    x = np.arange(len(profile))
    ax_prof.plot(x, profile,  color='lightgray', lw=0.8, label='raw')
    ax_prof.plot(x, smoothed, color='black',     lw=1.2, label='smoothed')
    ax_prof.scatter(peaks, smoothed[peaks], color='red', s=20, zorder=5, label='bands')
    ax_prof.axvline(centromere, color='cyan', lw=1.2, linestyle='--', label='centromere')
    ax_prof.set_xlim(0, len(profile) - 1)
    ax_prof.set_xlabel("row (px)", fontsize=7)
    ax_prof.set_ylabel("inverted intensity", fontsize=7)
    ax_prof.tick_params(labelsize=6)
    ax_prof.legend(fontsize=6, loc='upper right')

plt.suptitle("Chromosome band detection", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
p2, p98 = np.percentile(rotated, (2, 98))
rotated_stretched = exposure.rescale_intensity(rotated, in_range=(p2, p98))

display_image_histogram(rotated,           title="Before contrast stretching")
display_image_histogram(rotated_stretched, title="After contrast stretching")
